# 02 — Portuguese fuel prices vs Brent
Weekly pump prices for gasoline and diesel in Portugal (EU Weekly Oil Bulletin), with and without taxes.

In [1]:
# Import the libraries
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Open the Oil Bulletin Excel file and list its sheets
file = "../data/raw/oil_bulletin_history.xlsx"
excel = pd.ExcelFile(file)
print(excel.sheet_names)

['Prices with taxes', 'Prices wo taxes', 'Consumption', 'VAT', 'Excise duties', 'Excise duties - components', 'Other Indirect Taxes']


In [3]:
# Print the column names about Portugal in each sheet
for sheet in excel.sheet_names:
    data = pd.read_excel(file, sheet_name=sheet)
    for col in data.columns:
        if "PT" in str(col):
            print(sheet, "->", col)

Prices with taxes -> PT_price_with_tax_euro95
Prices with taxes -> PT_price_with_tax_diesel
Prices with taxes -> PT_price_with_tax_heating_oil
Prices with taxes -> PT_price_with_tax_fuel_oil_1
Prices with taxes -> PT_price_with_tax_fuel_oil_2
Prices with taxes -> PT_price_with_tax_LPG
Prices wo taxes -> PT_price_wo_tax_euro95
Prices wo taxes -> PT_price_wo_tax_diesel
Prices wo taxes -> PT_price_wo_tax_heating_oil
Prices wo taxes -> PT_price_wo_tax_fuel_oil_1
Prices wo taxes -> PT_price_wo_tax_fuel_oil_2
Prices wo taxes -> PT_price_wo_tax_LPG
Consumption -> PT_consumption_euro95
Consumption -> PT_consumption_diesel
Consumption -> PT_consumption_hePTing_oil
Consumption -> PT_consumption_fuel_oil_1
Consumption -> PT_consumption_fuel_oil_2
Consumption -> PT_consumption_LPG


## Clean Portuguese prices (euros per litre, from 2019)

In [5]:
# Names of the sheets and columns in the Excel file (check them in the output above)
SHEET_WITH_TAX = "Prices with taxes"
SHEET_WITHOUT_TAX = "Prices wo taxes"

# Load prices with taxes and keep the date, gasoline and diesel for Portugal
with_tax = pd.read_excel(file, sheet_name=SHEET_WITH_TAX)
with_tax = with_tax.rename(columns={
    with_tax.columns[0]: "date",
    "PT_price_with_tax_euro95": "gasoline_with_tax",
    "PT_price_with_tax_diesel": "diesel_with_tax",
})
with_tax = with_tax[["date", "gasoline_with_tax", "diesel_with_tax"]]

# Load prices without taxes and do the same
without_tax = pd.read_excel(file, sheet_name=SHEET_WITHOUT_TAX)
without_tax = without_tax.rename(columns={
    without_tax.columns[0]: "date",
    "PT_price_wo_tax_euro95": "gasoline_without_tax",
    "PT_price_wo_tax_diesel": "diesel_without_tax",
})
without_tax = without_tax[["date", "gasoline_without_tax", "diesel_without_tax"]]

# Merge both tables by date
fuel = pd.merge(with_tax, without_tax, on="date")

# Turn dates into real dates and prices into numbers (in euros per 1000 litres)
fuel["date"] = pd.to_datetime(fuel["date"], errors="coerce")
price_columns = ["gasoline_with_tax", "diesel_with_tax", "gasoline_without_tax", "diesel_without_tax"]
for col in price_columns:
    fuel[col] = pd.to_numeric(fuel[col], errors="coerce")

# Remove empty rows, convert to euros per litre and keep data from 2019 onwards
fuel = fuel.dropna()
for col in price_columns:
    fuel[col] = fuel[col] / 1000
fuel = fuel[fuel["date"] >= "2019-01-01"]
fuel = fuel.sort_values("date")

# Save the clean data and check it
fuel.to_csv("../data/processed/fuel_prices_pt.csv", index=False)
print(len(fuel), "weeks")
fuel.tail()

395 weeks


C:\Users\José\AppData\Local\Temp\ipykernel_9960\427312400.py:27: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fuel["date"] = pd.to_datetime(fuel["date"], errors="coerce")


,date,gasoline_with_tax,diesel_with_tax,gasoline_without_tax,diesel_without_tax
30,2026-08-31,2.013,2.029,1.031578,1.186283
29,2026-09-07,2.093,2.104,1.120478,1.272119
28,2026-09-14,2.072,2.152,1.093845,1.312233
27,2026-09-21,2.107,2.210,1.130600,1.370627
26,2026-09-28,2.099,2.186,1.123306,1.342585


## Pump prices vs Brent

In [6]:
# Load the weekly Brent data from notebook 01
brent = pd.read_csv("../data/processed/brent_eur_weekly.csv")
brent["date"] = pd.to_datetime(brent["date"])

# Move Brent dates from Sunday to the next Monday (pump prices on Monday reflect last week's oil price)
brent["date"] = brent["date"] + pd.Timedelta(days=1)

# Merge fuel prices and Brent by date
df = pd.merge(fuel, brent[["date", "brent_eur_litre"]], on="date")
print(len(df), "weeks with both fuel and Brent data")
df.tail()

395 weeks with both fuel and Brent data


,date,gasoline_with_tax,diesel_with_tax,gasoline_without_tax,diesel_without_tax,brent_eur_litre
390,2026-08-31,2.013,2.029,1.031578,1.186283,0.484528
391,2026-09-07,2.093,2.104,1.120478,1.272119,0.536991
392,2026-09-14,2.072,2.152,1.093845,1.312233,0.615192
393,2026-09-21,2.107,2.210,1.130600,1.370627,0.678115
394,2026-09-28,2.099,2.186,1.123306,1.342585,0.645257


In [7]:
# Load the weekly Brent data from notebook 01
brent = pd.read_csv("../data/processed/brent_eur_weekly.csv")
brent["date"] = pd.to_datetime(brent["date"])

# Move Brent dates from Sunday to the next Monday (pump prices on Monday reflect last week's oil price)
brent["date"] = brent["date"] + pd.Timedelta(days=1)

# Merge fuel prices and Brent by date
df = pd.merge(fuel, brent[["date", "brent_eur_litre"]], on="date")
print(len(df), "weeks with both fuel and Brent data")
df.tail()

395 weeks with both fuel and Brent data


,date,gasoline_with_tax,diesel_with_tax,gasoline_without_tax,diesel_without_tax,brent_eur_litre
390,2026-08-31,2.013,2.029,1.031578,1.186283,0.484528
391,2026-09-07,2.093,2.104,1.120478,1.272119,0.536991
392,2026-09-14,2.072,2.152,1.093845,1.312233,0.615192
393,2026-09-21,2.107,2.210,1.130600,1.370627,0.678115
394,2026-09-28,2.099,2.186,1.123306,1.342585,0.645257
